## Stage 0 (continued) — RSA check using real INTERSECTIONAL demographic cells

Follow-up to `pipeline/01_rsa_pilot/notebook.ipynb` (see
`notes/research_question/03_pivot2_group_consistency.md` §12.10). The previous
test only used **single-attribute** groups (e.g. just `"RACE: Asian"`) as a
proxy -- this one repeats the same methodology using real **intersectional**
cells (e.g. `"Black | Hindu"`), the actual target of `L_group` (the original
motivating example from Thread 2).

The intersectional cell data is built from raw individual Pew ATP responses
(15 waves) via `scripts/build_intersectional_cells.py` -- **169 cells** (only
163 raw labels are unique, but 6 of them happen to collide textually across
different combination types, e.g. `"Other | Other"` can arise from either
RACExPOLPARTY or RELIGxPOLPARTY -- which is why they are distinguished by
`attribute + group`, not by `group` alone), 6 attribute combinations
(`RACExRELIG`, `RACExPOLPARTY`, `RACExPOLIDEOLOGY`, `RELIGxPOLPARTY`,
`EDUCATIONxINCOME`, `AGExPOLPARTY`), with a minimum threshold of 30
respondents per cell per survey wave.

**What differs from the previous test:** previously there were only 2 kinds of
pair (same-attribute vs cross-attribute). Now each cell has 2 components, so
there are **3 kinds of pair**:

1. **Differing in just 1 component** (e.g. `"Black | Hindu"` vs `"Black | Muslim"`
   -- same race, different religion) -- this is the **most relevant** one for the
   real `L_group` use case (finding neighbours for a cell with little data).
2. **Differing in 2 components at once, but still within 1 combination type**
   (e.g. `"Black | Hindu"` vs `"White | Muslim"`, both RACExRELIG).
3. **Differing in combination type entirely** (e.g. RACExRELIG vs
   EDUCATIONxINCOME) -- the analogue of "cross-attribute" from the previous
   test, low priority.

## Before running: Kaggle settings

1. **Accelerator**: GPU T4 x2 or P100. **Internet: On**.
2. **Upload** `opinionqa_intersectional.csv`
   (original path: `datasets/subpop/data/opinionqa/processed/opinionqa_intersectional.csv`)
   as a Kaggle Dataset and attach it to this notebook -- the same way as for
   notebook 06 (Add Data -> Upload -> Create).

Estimated time: 15-25 minutes.

In [ ]:
!pip install -q -U "transformers>=4.44" accelerate scipy scikit-learn tqdm
!pip install -q -U bitsandbytes

In [ ]:
import os
import ast
import glob
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from scipy.stats import wasserstein_distance, spearmanr
from sklearn.metrics import pairwise_distances
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
else:
    raise RuntimeError(
        "No GPU detected. Check Notebook options -> Accelerator -> GPU T4 x2/P100, "
        "then restart & Run All again."
    )

In [ ]:
MODEL_PATH = "mistralai/Mistral-7B-v0.1"
USE_4BIT = False

_candidates = glob.glob("/kaggle/input/**/opinionqa_intersectional.csv", recursive=True)
if _candidates:
    DATA_PATH = _candidates[0]
elif os.path.exists("opinionqa_intersectional.csv"):
    DATA_PATH = "opinionqa_intersectional.csv"
else:
    raise FileNotFoundError(
        "Could not find opinionqa_intersectional.csv. Upload it as a Kaggle Dataset first, "
        "attach it, then re-run this cell. Or fill it in manually: DATA_PATH = '/kaggle/input/.../opinionqa_intersectional.csv'"
    )
print("Using data from:", DATA_PATH)

N_QKEYS_FOR_WD = 300  # sample of shared questions used to compute the real distance for each cell pair (for speed)
RANDOM_SEED = 42
N_PERMUTATIONS = 2000

OUT_DIR = "/kaggle/working/stage0_rsa_intersectional"
assert not OUT_DIR.startswith("/kaggle/input"), "OUT_DIR must be under /kaggle/working, not /kaggle/input!"
os.makedirs(OUT_DIR, exist_ok=True)
print("OUT_DIR:", OUT_DIR)

In [ ]:
df = pd.read_csv(DATA_PATH)

def _parse_list(x):
    return ast.literal_eval(x) if isinstance(x, str) else x

df["responses"] = df["responses"].apply(_parse_list)
df["ordinal"] = df["ordinal"].apply(_parse_list)
df["options"] = df["options"].apply(_parse_list)
df["group_key"] = df["attribute"] + " :: " + df["group"]

print(f"Total rows: {len(df)}")
print(f"Number of unique intersectional cells: {df['group_key'].nunique()}")
print(f"Number of unique questions (qkey): {df['qkey'].nunique()}")
print(f"Attribute combinations: {sorted(df['attribute'].unique().tolist())}")

## 1. Prepare the cell metadata: split into components (combination type, value-1, value-2)

Used to build the 3 pair masks below.

In [ ]:
GROUP_KEYS = sorted(df["group_key"].unique().tolist())
n_g = len(GROUP_KEYS)
print(f"{n_g} intersectional cells")

group_meta = {}
for gk in GROUP_KEYS:
    attr_type, group_str = gk.split(" :: ", 1)
    v1, v2 = group_str.split(" | ", 1)
    group_meta[gk] = {"attr_type": attr_type, "v1": v1, "v2": v2}

attr_types = np.array([group_meta[gk]["attr_type"] for gk in GROUP_KEYS])
v1_arr = np.array([group_meta[gk]["v1"] for gk in GROUP_KEYS])
v2_arr = np.array([group_meta[gk]["v2"] for gk in GROUP_KEYS])

same_type = attr_types[:, None] == attr_types[None, :]
share_v1 = v1_arr[:, None] == v1_arr[None, :]
share_v2 = v2_arr[:, None] == v2_arr[None, :]
share_exactly_one = same_type & (share_v1 ^ share_v2)
share_neither_same_type = same_type & (~share_v1) & (~share_v2)
diff_type = ~same_type

print("Number of pairs per kind (out of", n_g * (n_g - 1) // 2, "pairs in total):")
print("  differing in just 1 component        :", np.triu(share_exactly_one, k=1).sum())
print("  differing in 2 components, same type :", np.triu(share_neither_same_type, k=1).sum())
print("  differing in combination type        :", np.triu(diff_type, k=1).sum())

## 2. Compute the real distance between cells (Wasserstein Distance, sampled for speed)

In [ ]:
resp_lookup = {
    (gk, qk): (resp, ordv)
    for gk, qk, resp, ordv in zip(df["group_key"], df["qkey"], df["responses"], df["ordinal"])
}
qkeys_by_group = df.groupby("group_key")["qkey"].apply(set).to_dict()

rng = np.random.default_rng(RANDOM_SEED)
group_real_dist = np.full((n_g, n_g), np.nan)

for i in tqdm(range(n_g), desc="Computing real distances between intersectional cells"):
    gi = GROUP_KEYS[i]
    for j in range(i, n_g):
        if i == j:
            group_real_dist[i, j] = 0.0
            continue
        gj = GROUP_KEYS[j]
        shared = qkeys_by_group.get(gi, set()) & qkeys_by_group.get(gj, set())
        if not shared:
            continue
        shared = list(shared)
        if len(shared) > N_QKEYS_FOR_WD:
            idx = rng.choice(len(shared), size=N_QKEYS_FOR_WD, replace=False)
            shared = [shared[k] for k in idx]

        wds = []
        for qk in shared:
            respA, ordA = resp_lookup[(gi, qk)]
            respB, ordB = resp_lookup[(gj, qk)]
            if len(ordA) != len(ordB):
                continue
            wds.append(wasserstein_distance(ordA, ordB, u_weights=respA, v_weights=respB))
        if wds:
            group_real_dist[i, j] = group_real_dist[j, i] = float(np.mean(wds))

n_pairs_total = n_g * (n_g - 1) // 2
n_missing = int(np.isnan(group_real_dist[np.triu_indices(n_g, k=1)]).sum())
print(f"Done. Pairs with no shared question (real distance = NaN): {n_missing} out of {n_pairs_total}")
print("\nExample, the 6 cells most similar to the first cell:")
print(pd.Series(group_real_dist[0], index=GROUP_KEYS).sort_values().head(7))

## 3. Build a natural-sentence prompt per intersectional cell

The sentence format differs per combination type (to keep it natural), but
none of them mention internal survey attribute codes -- the lesson from
`06_stage0_rsa_kaggle.ipynb` (the `"ATTR: value"` format was already checked
and is not the culprit, but natural sentences remain best practice).

In [ ]:
def fmt_polideology(v):
    return v.lower()

PAIR_TEMPLATES = {
    "RACExRELIG": lambda v1, v2: f"This survey respondent's race is {v1} and their religion is {v2}.",
    "RACExPOLPARTY": lambda v1, v2: f"This survey respondent's race is {v1} and their political party affiliation is {v2}.",
    "RACExPOLIDEOLOGY": lambda v1, v2: f"This survey respondent's race is {v1}, and politically they describe their views as {fmt_polideology(v2)}.",
    "RELIGxPOLPARTY": lambda v1, v2: f"This survey respondent's religion is {v1} and their political party affiliation is {v2}.",
    "EDUCATIONxINCOME": lambda v1, v2: f"This survey respondent's highest level of education is {v1}, and their household income is {v2}.",
    "AGExPOLPARTY": lambda v1, v2: f"This survey respondent is {v1} years old and their political party affiliation is {v2}.",
}

def build_prompt(gk):
    meta = group_meta[gk]
    return PAIR_TEMPLATES[meta["attr_type"]](meta["v1"], meta["v2"])

group_prompts = {gk: build_prompt(gk) for gk in GROUP_KEYS}
print("Example prompt per combination type:\n")
seen_types = set()
for gk in GROUP_KEYS:
    t = group_meta[gk]["attr_type"]
    if t not in seen_types:
        print(f"[{t}] {group_prompts[gk]!r}")
        seen_types.add(t)

## 4. Load the model & extract the representations

In [ ]:
print(f"Loading tokenizer & model: {MODEL_PATH}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model_kwargs = dict(torch_dtype=torch.float16, device_map="auto", low_cpu_mem_usage=True)
if USE_4BIT:
    from transformers import BitsAndBytesConfig
    model_kwargs.pop("torch_dtype", None)
    model_kwargs["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)

model = AutoModelForCausalLM.from_pretrained(MODEL_PATH, **model_kwargs)
model.eval()

NUM_LAYERS = model.config.num_hidden_layers
print(f"Model loaded. Number of layers: {NUM_LAYERS} (+1 initial embedding)")

In [ ]:
@torch.no_grad()
def get_hidden_states_all_layers(prompt: str) -> np.ndarray:
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model(**inputs, output_hidden_states=True)
    hs = torch.stack(out.hidden_states, dim=0)
    return hs[:, 0, -1, :].float().cpu().numpy()

group_embeddings = {}
for gk, prompt in tqdm(group_prompts.items(), desc="Extracting intersectional cell representations"):
    group_embeddings[gk] = get_hidden_states_all_layers(prompt)
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

group_emb_array = np.stack([group_embeddings[gk] for gk in GROUP_KEYS])
n_layers_total = group_emb_array.shape[1]

np.savez(
    os.path.join(OUT_DIR, "embeddings_intersectional.npz"),
    group_emb=group_emb_array,
    group_keys=np.array(GROUP_KEYS, dtype=object),
)
print("Representations saved to", os.path.join(OUT_DIR, "embeddings_intersectional.npz"))

## 5. RSA per kind of pair, per layer

Same as before: the Spearman correlation between the LLM-representation
distance vs the real distance, plus a permutation test (Mantel test) at the
best layer.

In [ ]:
def upper_tri(mat):
    idx = np.triu_indices_from(mat, k=1)
    return mat[idx]

def rho_masked(emb_array, real_dist_matrix, layer_idx, pair_mask):
    rep_dist = pairwise_distances(emb_array[:, layer_idx, :], metric="cosine")
    idx = np.triu_indices_from(real_dist_matrix, k=1)
    keep = pair_mask[idx]
    real_flat, rep_flat = real_dist_matrix[idx][keep], rep_dist[idx][keep]
    valid = ~np.isnan(real_flat) & ~np.isnan(rep_flat)
    if valid.sum() < 5:
        return np.nan
    rho, _ = spearmanr(real_flat[valid], rep_flat[valid])
    return rho

def rsa_perm_masked(emb_array, real_dist_matrix, layer_idx, pair_mask, n_perm=N_PERMUTATIONS, seed=RANDOM_SEED):
    rep_dist = pairwise_distances(emb_array[:, layer_idx, :], metric="cosine")
    idx = np.triu_indices_from(real_dist_matrix, k=1)
    keep = pair_mask[idx]
    real_flat, rep_flat = real_dist_matrix[idx][keep], rep_dist[idx][keep]
    valid = ~np.isnan(real_flat) & ~np.isnan(rep_flat)
    real_flat, rep_flat = real_flat[valid], rep_flat[valid]
    rho, _ = spearmanr(real_flat, rep_flat)

    rng_p = np.random.default_rng(seed)
    n = real_dist_matrix.shape[0]
    perm_rhos = np.empty(n_perm)
    for k in range(n_perm):
        perm = rng_p.permutation(n)
        permuted_flat = real_dist_matrix[np.ix_(perm, perm)][idx][keep][valid]
        r, _ = spearmanr(permuted_flat, rep_flat)
        perm_rhos[k] = 0.0 if np.isnan(r) else r
    return float(rho), float(np.mean(np.abs(perm_rhos) >= abs(rho)))

MASKS = {
    "diff_1_component": share_exactly_one,
    "diff_2_components_same_type": share_neither_same_type,
    "diff_combination_type": diff_type,
}

rho_table = {"layer": list(range(n_layers_total))}
for name, mask in MASKS.items():
    rho_table[name] = [rho_masked(group_emb_array, group_real_dist, L, mask) for L in range(n_layers_total)]

rho_df = pd.DataFrame(rho_table)
print(rho_df.to_string())

In [ ]:
summary_rows = []
for name, mask in MASKS.items():
    rhos = rho_df[name].values
    best_layer = int(np.nanargmax(rhos))  # look for the MOST POSITIVE, not the abs -- we want the correct direction
    rho_best, p_best = rsa_perm_masked(group_emb_array, group_real_dist, best_layer, mask)
    summary_rows.append({"pair_kind": name, "best_layer": best_layer, "rho": rho_best, "p_value": p_best,
                          "n_pairs": int(np.triu(mask, k=1).sum())})
    print(f"[{name}] best layer {best_layer}: rho={rho_best:.3f}, p={p_best:.4f} (n={int(np.triu(mask, k=1).sum())} pairs)")

summary_df = pd.DataFrame(summary_rows)

In [ ]:
print("--- breakdown of 'diff_1_component' PER COMBINATION TYPE (not pooled over the 6 types) ---")
print("(a pooled rho over 719 pairs can hide one weak type when the others are strong -- check RACExRELIG on its own)")
for t in sorted(set(attr_types)):
    type_mask = share_exactly_one & (attr_types[:, None] == t) & (attr_types[None, :] == t)
    n_pairs = int(np.triu(type_mask, k=1).sum())
    if n_pairs < 5:
        print(f"{t}: only {n_pairs} pairs, skipping")
        continue
    rhos = [rho_masked(group_emb_array, group_real_dist, L, type_mask) for L in range(n_layers_total)]
    best_layer = int(np.nanargmax(rhos))
    rho_best, p_best = rsa_perm_masked(group_emb_array, group_real_dist, best_layer, type_mask)
    print(f"{t}: layer {best_layer}, rho={rho_best:.3f}, p={p_best:.4f}, n={n_pairs}")

## 5b. Digging into the embedding geometry (track: go deeper on the embeddings)

The RSA above only tells us the *ranking* is reasonable (rho ~0.17), but it
does NOT tell us **how wide** the spread of the embeddings is. rho could be
positive even while all the cells are crammed together (rho is rank-based, it
does not care about absolute distances). The three checks below are all cheap
-- just numpy on the `group_emb_array` we already have:

- **Step 1 — distance spread:** min/median/max cosine distance between cells at
  the best layer. Directly answers: "could it be that everything is similar?"
- **Step 2 — anisotropy + mean-centering:** is there a shared dominant
  direction covering up the signal? If rho RISES after centering → the signal
  was merely buried and can be dug out. If it stays put → that really is all
  there is.
- **Step 3 — per type:** already in the cell above (RACExRELIG isolated).

In [ ]:
# === STEP 1: how wide is the spread of distances between cells? (best layer for beda_1) ===
BEST = int(summary_df.loc[summary_df["pair_kind"] == "diff_1_component", "best_layer"].iloc[0])
emb_best = group_emb_array[:, BEST, :]
d = pairwise_distances(emb_best, metric="cosine")
iu = np.triu_indices(n_g, k=1)
alld = d[iu]

print(f"Best layer (diff_1_component) = {BEST}. Spread of cosine distances between {n_g} cells:\n")
print(f"  ALL {len(alld)} pairs : min={alld.min():.4f}  median={np.median(alld):.4f}  "
      f"max={alld.max():.4f}  std={alld.std():.4f}")
print(f"  spread ratio (max-min)/mean = {(alld.max()-alld.min())/alld.mean():.3f}")
print(f"  -> cosine SIMILARITY: even the MOST DIFFERENT pair is still {1-alld.max():.3f} similar")

# split per kind of pair so the same-type vs different-type contrast is visible
for name, mask in MASKS.items():
    sub = d[iu][mask[iu]]
    sub = sub[~np.isnan(sub)]
    if len(sub):
        print(f"  [{name}] min={sub.min():.4f} median={np.median(sub):.4f} max={sub.max():.4f} (n={len(sub)})")

print("\n  How to read this: if the max is far below 1 (e.g. <0.1) and the spread ratio is small,")
print("  then all the cells are piled into one narrow cone -> 'everything is similar' really is happening.")

fig, ax = plt.subplots(figsize=(7, 4))
for name, mask in MASKS.items():
    sub = d[iu][mask[iu]]
    sub = sub[~np.isnan(sub)]
    if len(sub):
        ax.hist(sub, bins=40, alpha=0.55, label=f"{name} (n={len(sub)})")
ax.set_xlabel(f"cosine distance between cells (layer {BEST})")
ax.set_ylabel("number of pairs")
ax.set_title(f"Step 1 -- spread of embedding distances between cells (layer {BEST})")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "step1_distance_distribution.png"), dpi=150)
plt.show()

In [ ]:
# === STEP 2: anisotropy (shared dominant direction) + the effect of mean-centering ===

# 2a. How "aligned" are all the embeddings at the best layer?
mean_vec = emb_best.mean(axis=0)
norm_mean = np.linalg.norm(mean_vec)
mean_norm = np.linalg.norm(emb_best, axis=1).mean()
print(f"=== STEP 2a: anisotropy at layer {BEST} ===")
print(f"  ||mean vector|| / mean ||vector|| = {norm_mean/mean_norm:.3f}")
print("  (close to 1 = every vector points in the same single direction = severely anisotropic;")
print("   close to 0 = the directions are spread all over)")

# PCA on the ALREADY-centered embeddings: is the remaining variation low-rank or not?
Xc = emb_best - mean_vec
s = np.linalg.svd(Xc, compute_uv=False)
evr = (s**2) / (s**2).sum()
print(f"\n  After centering, share of the remaining variation taken by each component (PCA):")
print(f"  PC1={evr[0]:.1%}, PC2={evr[1]:.1%}, PC3={evr[2]:.1%}, "
      f"top 5 total={evr[:5].sum():.1%}")

# 2b. THE CORE: does mean-centering RAISE rho? (was the signal buried under a shared direction?)
# center per layer (subtract the mean across cells at each layer), then re-measure the RSA.
group_emb_centered = group_emb_array - group_emb_array.mean(axis=0, keepdims=True)

print(f"\n=== STEP 2b: rho BEFORE vs AFTER mean-centering (diff_1_component) ===")
print(f"  {'layer':>5} | {'rho raw':>9} | {'rho center':>10} | delta")
best_gain = (None, -9)
for L in range(n_layers_total):
    r0 = rho_masked(group_emb_array, group_real_dist, L, share_exactly_one)
    r1 = rho_masked(group_emb_centered, group_real_dist, L, share_exactly_one)
    if not np.isnan(r0):
        tag = "  <-- up" if (r1 - r0) > 0.02 else ""
        print(f"  {L:>5} | {r0:>+9.3f} | {r1:>+10.3f} | {r1-r0:>+.3f}{tag}")
        if r1 > best_gain[1]:
            best_gain = (L, r1)

print(f"\n  highest rho after centering: layer {best_gain[0]}, rho={best_gain[1]:+.3f}")
print("  -> if the 'rho center' column is consistently HIGHER than 'rho raw', it means")
print("     the demographic signal really was buried under a shared dominant direction --")
print("     mean-centering (or whitening) is worth applying before building the kernel. If it")
print("     does not rise, the problem is elsewhere; look somewhere else (prompt/pooling).")

## 5c. Combining the findings: per type AFTER centering + precision@k

Two follow-up checks (tying findings 02 & 03 together):

- **Step 4 — per type × centering:** does mean-centering rescue the weak
  RACExRELIG? Do AGE/EDU break above 0.50?
- **Step 5 — precision@k:** the global rho ranks ALL pairs, but the `L_group`
  kernel only uses the k nearest neighbours. Check it directly: of the k
  nearest neighbours according to the embeddings, how many are genuinely among
  the k nearest according to the survey? Compared against chance (random
  guessing) so the number means something.

In [ ]:
# === STEP 4: per type, rho BEFORE vs AFTER mean-centering (diff_1_component) ===
import warnings
warnings.filterwarnings("ignore")  # spearmanr ConstantInputWarning on small subsets, not important

print(f"{'type':<18} | {'rho raw':>8} @L | {'rho center':>10} @L | n pairs")
print("-" * 60)
for t in sorted(set(attr_types)):
    type_mask = share_exactly_one & (attr_types[:, None] == t) & (attr_types[None, :] == t)
    n_pairs = int(np.triu(type_mask, k=1).sum())
    if n_pairs < 5:
        print(f"{t:<18} | only {n_pairs} pairs, skipping")
        continue
    rr = [rho_masked(group_emb_array, group_real_dist, L, type_mask) for L in range(n_layers_total)]
    rc = [rho_masked(group_emb_centered, group_real_dist, L, type_mask) for L in range(n_layers_total)]
    Lr, Lc = int(np.nanargmax(rr)), int(np.nanargmax(rc))
    print(f"{t:<18} | {rr[Lr]:>+8.3f} {Lr:>2} | {rc[Lc]:>+10.3f} {Lc:>2} | {n_pairs}")

print("\n  -> look especially at RACExRELIG: if its center column jumps a long way, the")
print("     signal was merely buried (not absent). If it stays low, it really is weak.")

In [ ]:
# === STEP 5: precision@k -- are the top-k neighbours according to the embeddings accurate? ===
K = 5

def precision_at_k(emb_layer, subset, k=K):
    """subset = indices of the cells of one type. For each cell: of the k nearest
    according to the embedding, how many are also k-nearest according to the survey?
    Averaged over cells."""
    sub = np.array(subset)
    ed = pairwise_distances(emb_layer[sub], metric="cosine")
    rd = group_real_dist[np.ix_(sub, sub)].copy()
    np.fill_diagonal(ed, np.inf)
    np.fill_diagonal(rd, np.inf)
    precs = []
    for i in range(len(sub)):
        valid = ~np.isnan(rd[i])
        if valid.sum() < k:
            continue
        emb_nn = set(np.argsort(ed[i])[:k])
        surv_nn = set(np.argsort(np.where(valid, rd[i], np.inf))[:k])
        precs.append(len(emb_nn & surv_nn) / k)
    return (float(np.mean(precs)), len(precs)) if precs else (np.nan, 0)

print(f"precision@{K} (within type): how many of the {K} nearest-neighbours-by-embedding")
print(f"are genuinely the {K} nearest according to the survey. Compare raw vs center vs chance.\n")
print(f"{'type':<18} | {'raw':>6} | {'center':>6} | {'chance':>6} | m cells")
print("-" * 55)
for t in sorted(set(attr_types)):
    subset = [i for i in range(n_g) if attr_types[i] == t]
    m = len(subset)
    if m < K + 2:
        print(f"{t:<18} | only {m} cells, skipping")
        continue
    # pick the best layer for this type (centered version), and use the same layer for raw
    tm = share_exactly_one & (attr_types[:, None] == t) & (attr_types[None, :] == t)
    rc = [rho_masked(group_emb_centered, group_real_dist, L, tm) for L in range(n_layers_total)]
    Lc = int(np.nanargmax(rc))
    pr_raw, _ = precision_at_k(group_emb_array[:, Lc, :], subset)
    pr_cen, nsel = precision_at_k(group_emb_centered[:, Lc, :], subset)
    chance = K / (m - 1)
    print(f"{t:<18} | {pr_raw:>6.2f} | {pr_cen:>6.2f} | {chance:>6.2f} | {m}")

print(f"\n  -> 'center' well ABOVE 'chance' = the top-{K} kernel really is informative")
print(f"     (this matters most for L_group, more than the global rho).")
print(f"     'center' merely on a par with 'chance' = even if rho looks fine, the top-{K} neighbours are useless.")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for name in MASKS:
    ax.plot(rho_df["layer"], rho_df[name], marker="o", label=name)
ax.axhline(0, color="gray", linewidth=0.8)
ax.axvline(15, color="red", linestyle="--", alpha=0.4, label="layer ~15 (llm-opinions threshold)")
ax.set_xlabel("Layer")
ax.set_ylabel("RSA Spearman rho")
ax.set_title("Stage 0 (intersectional cells) -- RSA per kind of pair, per layer")
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "rsa_intersectional_per_layer.png"), dpi=150)
plt.show()

rho_df.to_csv(os.path.join(OUT_DIR, "rsa_intersectional_rho_per_layer.csv"), index=False)
summary_df.to_csv(os.path.join(OUT_DIR, "rsa_intersectional_summary.csv"), index=False)
print(summary_df.to_string(index=False))
print(f"\nOutputs are in: {OUT_DIR}")

## 6. Look at it directly: Map 1 (real distance) vs Map 2 (distance inside the LLM's head)

This is the heart of Stage 0 -- not just the rho number, but genuinely
**matching up two distance maps** for the same pairs:

- **Map 1** = the real distance from the survey data (`group_real_dist`,
  already computed in section 2).
- **Map 2** = the LLM representation distance (cosine distance between
  embeddings, at the best layer found for `diff_1_component`).

Each point in the scatter plot below = **1 pair of cells**. The X axis = Map 1,
the Y axis = Map 2. If the points slope upwards from bottom-left to top-right
(a line tilted upwards), the two maps line up (this is what gets summarised as
a positive rho). If they are scattered with no direction, they do not line up.

In [ ]:
BEST_LAYER_FOR_PLOT = int(summary_df.loc[summary_df["pair_kind"] == "diff_1_component", "best_layer"].iloc[0])
rep_dist_at_best = pairwise_distances(group_emb_array[:, BEST_LAYER_FOR_PLOT, :], metric="cosine")

colors = {
    "diff_1_component": "#1f77b4",
    "diff_2_components_same_type": "#ff7f0e",
    "diff_combination_type": "#7f7f7f",
}
idx_ut = np.triu_indices(n_g, k=1)

fig, ax = plt.subplots(figsize=(7, 6))
for name, mask in MASKS.items():
    keep = mask[idx_ut]
    x = group_real_dist[idx_ut][keep]
    y = rep_dist_at_best[idx_ut][keep]
    valid = ~np.isnan(x) & ~np.isnan(y)
    alpha = 0.35 if name == "diff_combination_type" else 0.6
    ax.scatter(x[valid], y[valid], s=10, alpha=alpha, label=f"{name} (n={int(valid.sum())})", color=colors[name])

ax.set_xlabel("Map 1: real distance from the survey (Wasserstein Distance)")
ax.set_ylabel(f"Map 2: LLM representation distance (cosine, layer {BEST_LAYER_FOR_PLOT})")
ax.set_title("Stage 0 (intersectional cells) -- Map 1 vs Map 2, each point = 1 pair of cells")
ax.legend(markerscale=2, fontsize=8)
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "rsa_scatter_map1_vs_map2.png"), dpi=150)
plt.show()

To make it more concrete, here are a few **real** pairs of cells from our data
(not hypothetical ones) -- Map 1 and Map 2 printed side by side:

In [ ]:
def show_pair(gk1, gk2, label):
    if gk1 not in GROUP_KEYS or gk2 not in GROUP_KEYS:
        print(f"[{label}] one of the cells is missing from the data ({gk1} / {gk2}), skipping.\n")
        return
    i, j = GROUP_KEYS.index(gk1), GROUP_KEYS.index(gk2)
    print(f"[{label}]")
    print(f"  {gk1}")
    print(f"  {gk2}")
    print(f"  Map 1 (real distance, WD)                    = {group_real_dist[i, j]:.4f}")
    print(f"  Map 2 (LLM representation distance, layer {BEST_LAYER_FOR_PLOT}) = {rep_dist_at_best[i, j]:.4f}")
    print()

show_pair("RACExRELIG :: Asian | Hindu", "RACExRELIG :: Asian | Protestant",
          "differing in 1 component -- same RACE (Asian), different RELIG")
show_pair("RACExRELIG :: White | Protestant", "RACExRELIG :: Black | Protestant",
          "differing in 1 component -- same RELIG (Protestant), different RACE")
show_pair("RACExRELIG :: Asian | Hindu", "RACExRELIG :: White | Atheist",
          "differing in 2 components, same type (different RACE AND different RELIG)")
show_pair("RACExRELIG :: Asian | Hindu", "EDUCATIONxINCOME :: Associate's degree | $50,000-$75,000",
          "differing in combination type entirely")

## How to read the results

The most important thing: the **`diff_1_component`** row -- this is the closest
match to the real `L_group` use case (finding neighbours for a cell with little
data, differing in just 1 trait). If its rho is positive & significant, that is
strong confirmation that the finding from the earlier single-attribute test
(§12.10) does also hold for real intersectional cells, and was not just an
accident of the proxy data's structure.

The scatter plot & the concrete example pairs in section 6 are the **direct
visual evidence** behind that rho number -- if the blue points
(`diff_1_component`) visibly slope up from bottom-left to top-right compared to
the grey points (`diff_combination_type`), which are scattered or slope down,
that is the most concrete way to see "do these two maps line up or not" without
having to take the rho number on faith.

**Next steps:** download `/kaggle/working/stage0_rsa_intersectional/`
(which now includes `rsa_scatter_map1_vs_map2.png`), compare the numbers
against the table in `notes/research_question/03_pivot2_group_consistency.md`
§12.10/§12.11, then update that document if the scatter plot / concrete
examples turn up anything new.